In [ ]:
# [Setup]
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/code')

!pip install -q z3-solver evaluate python-Levenshtein bert_score "torchao>=0.16.0" 2>/dev/null
import nltk
nltk.download('punkt_tab', quiet=True)

from src.prepare_scorer_data import (
    step1_z3_filter,
    step2_n_unique,
    step3_z3_le_count,
    step4_mean_bleu,
    step5_mean_bertscore,
    step6_dedup_backtrans,
    step7_z3_le_label,
)

print('Ready.')

In [ ]:
# [CONFIG]
MODEL = "Qwen8b"  # "Qwen4b" | "Qwen8b" | "Ministral8b"
FORCE = False     # Set to True to recompute existing steps

print(f"Model:  {MODEL}")
print(f"Force:  {FORCE}")

In [ ]:
# [Step 1/7] Z3 filter — keep only candidates whose FOL can be parsed by Z3
# Reads:  {MODEL}_k10_val.json
# Writes: {MODEL}_val_scorer_step1.json

step1_z3_filter(model=MODEL, force=FORCE)

In [ ]:
# [Step 2/7] n_unique — count unique FOLs among Z3-filtered candidates per sentence
# Reads:  {MODEL}_val_scorer_step1.json
# Writes: {MODEL}_val_scorer_step2.json

step2_n_unique(model=MODEL, force=FORCE)

In [ ]:
# [Step 3/7] z3_le_count — pairwise Z3 LE among filtered candidates (before dedup)
# Reads:  {MODEL}_val_scorer_step2.json
# Writes: {MODEL}_val_scorer_step3.json

step3_z3_le_count(model=MODEL, force=FORCE)

In [ ]:
# [Step 4/7] mean_bleu — mean pairwise FOL-token BLEU among FILTERED candidates only
# Reads:  {MODEL}_val_scorer_step3.json
# Writes: {MODEL}_val_scorer_step4.json

step4_mean_bleu(model=MODEL, force=FORCE)

In [ ]:
# [Step 5/7] mean_bertscore — mean pairwise BertScore F1 among FILTERED candidates only
# Reads:  {MODEL}_val_scorer_step4.json
# Writes: {MODEL}_val_scorer_step5.json

step5_mean_bertscore(model=MODEL, force=FORCE)

In [ ]:
# [Step 6/7] backtrans_sim — dedup per sentence, then back-translate + cosine sim
# Deduplicates FOLs globally so each unique formula is informalized only ONCE.
# Uses persistent cache: {MODEL}_backtrans_cache.json (crash-safe resume).
# Reads:  {MODEL}_val_scorer_step5.json
# Writes: {MODEL}_val_scorer_step6.json

step6_dedup_backtrans(model=MODEL, force=FORCE)

In [ ]:
# [Step 7/7] z3_le_label — Z3 equivalence check: each candidate FOL vs ground-truth FOL
# This is the final label! Drops nl/gt_fol/fol, keeps only the 8 feature columns.
# Reads:  {MODEL}_val_scorer_step6.json
# Writes: {MODEL}_val_scorer.json (FINAL)

step7_z3_le_label(model=MODEL, force=FORCE)

In [ ]:
# [Validation] Load final output and inspect
import json, os

out_path = f"/content/drive/MyDrive/code/data/results/{MODEL}/k10/{MODEL}_val_scorer.json"

if os.path.exists(out_path):
    with open(out_path) as f:
        data = json.load(f)
    print(f"Total rows: {len(data)}")
    print(f"Columns: {list(data[0].keys())}")
    print()
    labels = [r['z3_le_label'] for r in data]
    n_pos = sum(labels)
    print(f"Label=1: {n_pos} ({100*n_pos/len(labels):.1f}%)")
    print(f"Label=0: {len(labels)-n_pos} ({100*(len(labels)-n_pos)/len(labels):.1f}%)")
    print()
    print("Sample rows:")
    for row in data[:5]:
        print(f"  {row}")
else:
    print(f"Output not found: {out_path}")
    print("Run all 7 steps above first.")